# Nominally significant univariate results

Read all univariate Cox results for the **ARPI** and **ADT** cohort arms and
retain rows with nominal significance (`p_value < 0.05`). This is an exploratory
filter; `q_value`
remains in the table so multiplicity-adjusted significance can be assessed
separately.

The shared-canonical result file is preferred because it contains every
reported landmark in one table and is the source used by Figure 3. Legacy
per-landmark files are loaded only when the shared file is unavailable.

In [ ]:
from pathlib import Path
from typing import List, Tuple
import re

import pandas as pd
from IPython.display import display

COHORT_LABELS = {"arpi": "ARPI", "adt": "ADT"}
NOMINAL_ALPHA = 0.05
NEPC_PROJ_PATH = Path("/data/gusev/USERS/jpconnor/data/CAIA/COMPASS")
SURVIVAL_DIR = NEPC_PROJ_PATH / "survival_analysis"
RUN_DIRS = {
    cohort: SURVIVAL_DIR / f"local_runs_{cohort}"
    for cohort in COHORT_LABELS
}

RESULT_FILENAME = "cox_agg_univariate_nobs_adjusted.csv"
EXPORT_PATHS = {
    cohort: run_dir / "cox" / "nominally_significant_univariate_results.csv"
    for cohort, run_dir in RUN_DIRS.items()
}

for cohort, run_dir in RUN_DIRS.items():
    print(f"{COHORT_LABELS[cohort]} ({cohort}): {run_dir}")

## Load every landmark

Using the shared file when present prevents double-counting the same models
from both shared and legacy per-landmark output directories.

In [ ]:
def load_univariate_results(
    run_dir: Path,
    cohort: str,
) -> Tuple[pd.DataFrame, List[Path]]:
    shared_path = run_dir / "cox" / "landmark_shared" / RESULT_FILENAME
    if shared_path.exists():
        paths = [shared_path]
    else:
        paths = sorted((run_dir / "cox").glob(
            f"landmark_*/both/{RESULT_FILENAME}"
        ))

    if not paths:
        raise FileNotFoundError(
            f"No univariate result files found under {run_dir / 'cox'}. "
            "Run the univariate models first."
        )

    frames = []
    for path in paths:
        frame = pd.read_csv(path)
        if "landmark_days" not in frame.columns:
            match = re.search(r"landmark_(-?\\d+)", str(path.parent.parent))
            if match is None:
                raise ValueError(f"Could not infer landmark from {path}")
            frame.insert(0, "landmark_days", int(match.group(1)))
        frame["source_path"] = str(path)
        frames.append(frame)

    results = pd.concat(frames, ignore_index=True)
    required = {"landmark_days", "endpoint", "feature", "p_value"}
    missing = required - set(results.columns)
    if missing:
        raise ValueError(f"Univariate results are missing columns: {sorted(missing)}")

    results["p_value"] = pd.to_numeric(results["p_value"], errors="coerce")
    results.insert(0, "cohort", cohort)
    return results, paths


results_by_cohort = {}
source_paths_by_cohort = {}
for cohort, run_dir in RUN_DIRS.items():
    results, source_paths = load_univariate_results(run_dir, cohort)
    results_by_cohort[cohort] = results
    source_paths_by_cohort[cohort] = source_paths
    print(f"\nLoaded {COHORT_LABELS[cohort]}:")
    for path in source_paths:
        print(f"  {path}")
    print(
        f"{len(results):,} rows across landmarks "
        f"{sorted(results['landmark_days'].dropna().unique().tolist())}"
    )

## Filter at nominal significance

No stability or false-discovery-rate filter is applied here: every finite
`p_value < 0.05` row is retained.

In [ ]:
preferred_columns = [
    "cohort", "landmark_days", "endpoint", "feature", "lab_name",
    "feature_stat", "coverage", "n_patients_used", "n_events_used",
    "coef_feature", "hazard_ratio_per_sd", "ci_lower", "ci_upper",
    "p_value", "q_value", "note", "model_type", "source_path",
]

def filter_nominal(results: pd.DataFrame) -> pd.DataFrame:
    filtered = (
        results.loc[
            results["p_value"].notna()
            & results["p_value"].lt(NOMINAL_ALPHA)
        ]
        .sort_values(["endpoint", "landmark_days", "p_value", "feature"])
        .reset_index(drop=True)
    )
    ordered = [c for c in preferred_columns if c in filtered.columns]
    ordered += [c for c in filtered.columns if c not in ordered]
    return filtered[ordered]


nominal_by_cohort = {
    cohort: filter_nominal(results)
    for cohort, results in results_by_cohort.items()
}
summary = (
    pd.concat(nominal_by_cohort.values(), ignore_index=True)
    .groupby(["cohort", "endpoint", "landmark_days"], dropna=False)
    .size()
    .rename("n_nominally_significant")
    .reset_index()
)

for cohort, filtered in nominal_by_cohort.items():
    print(
        f"{COHORT_LABELS[cohort]}: retained {len(filtered):,} / "
        f"{len(results_by_cohort[cohort]):,} rows with p < {NOMINAL_ALPHA}."
    )
display(summary)

## Review all nominal hits

In [ ]:
with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.width", 180,
):
    for cohort, filtered in nominal_by_cohort.items():
        print(f"\n{COHORT_LABELS[cohort]} nominal hits")
        display(filtered)

## Export one filtered table per cohort

In [ ]:
for cohort, filtered in nominal_by_cohort.items():
    export_path = EXPORT_PATHS[cohort]
    export_path.parent.mkdir(parents=True, exist_ok=True)
    filtered.to_csv(export_path, index=False)
    print(
        f"Wrote {len(filtered):,} {COHORT_LABELS[cohort]} rows to "
        f"{export_path}"
    )